# 01 — Fetch & Explore

This notebook fetches raw Mapillary speed sign detections and image sequences
and Overture road segments (which include normalized speed limit values) for a
small bounding box around Bountiful, UT.  Use it to:

* Explore data density (sign count, sequence count, road coverage)
* Get a quick overview map of the raw data
* Tune the bounding box before running the full pipeline

**Pre-requisites:**
```bash
pip install speed-limit-conflation
export MAPILLARY_ACCESS_TOKEN='<your token>'
```

In [ ]:
import os
import geopandas as gpd
from slc import fetch, viz

TOKEN = os.environ['MAPILLARY_ACCESS_TOKEN']

# Bountiful, UT — adjust as needed
BBOX = (-111.920, 40.855, -111.855, 40.910)  # (min_lon, min_lat, max_lon, max_lat)

In [ ]:
# Fetch signs
signs = fetch.fetch_mapillary_signs(BBOX, TOKEN)
print(f'Signs: {len(signs)}')
signs.head()

In [ ]:
# Fetch images and build sequences
images = fetch.fetch_mapillary_images(BBOX, TOKEN)
sequences = fetch.build_sequences(images)
print(f'Images: {len(images)}  |  Sequences: {len(sequences)}')
sequences.head()

In [ ]:
# Fetch Overture segments and parse their built-in speed limits
overture_raw = fetch.fetch_overture_segments(BBOX)
overture = fetch.extract_overture_speed_limits(overture_raw)
print(f'Overture segments: {len(overture)}')
print(f'Segments with speed limit: {overture["speed_limit_value"].notna().sum()}')
overture[['id', 'speed_limit_value']].head()

In [ ]:
# Quick stats
print('Mapillary sign speed distribution:')
print(signs['speed_mph'].value_counts())
print('\nOverture speed limit distribution:')
print(overture['speed_limit_value'].value_counts(dropna=False))

In [ ]:
# Overview map
viz.map_signs_and_sequences(signs, sequences)